# Autograd를 사용한 사용자 정의 활성화 함수 작성하기

### 문제 설명
`torch.autograd.Function`을 사용해 **Learned-SiLU**라는 **사용자 정의 활성화 함수**를 구현합니다. 이 활성화 함수는 SiLU 공식 \( x \cdot \text{sigmoid}(x) \) 을 기반으로 하지만, **학습 가능한 기울기(slope) 파라미터**를 포함합니다. 이 사용자 정의 활성화 함수를 간단한 선형 회귀 모델에 사용합니다.

### 요구사항
1. **사용자 정의 활성화 함수 정의**:
   - 출력이 다음과 같이 계산되는 사용자 정의 활성화 함수 **Learned-SiLU**를 구현합니다:
     $$
     \text{Learned-SiLU}(x) = \text{slope} \cdot x \cdot \text{sigmoid}(x)
     $$
   - **slope**는 학습 가능한 파라미터여야 합니다.

2. **Autograd 구현**:
   - `torch.autograd.Function`을 사용해 사용자 정의 활성화 함수의 forward와 backward 연산을 정의합니다.

3. **활성화 함수 통합**:
   - 이 사용자 정의 활성화 함수를 간단한 선형 회귀 모델에 통합합니다.
   - 모델을 학습해 활성화 함수가 제대로 동작하는지 검증합니다.

### 제약 사항
- **slope 파라미터**가 학습 중 올바르게 초기화되고 업데이트되어야 합니다.

<details>
  <summary>💡 힌트</summary>
  추가 설명: https://pytorch.org/tutorials/beginner/examples_autograd/two_layer_net_custom_function.html
</details>

<details>
  <summary>💡 대안 구현?</summary>
  backward를 구현하지 않고 `nn.Module`로도 구현할 수 있습니다.
</details>


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# Generate synthetic data
torch.manual_seed(42)
X = torch.rand(100, 1) * 10  # 100 data points between 0 and 10
y = 2 * X + 3 + torch.randn(100, 1)  # Linear relationship with noise

In [3]:
class LearnedSiLUFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, slope):
        # Save the input tensor and slope for backward computation
        ctx.save_for_backward(x, slope)
        return slope * x * torch.sigmoid(x)

    @staticmethod
    def backward(ctx, grad_output):
        # Retrieve the input and slope saved in the forward pass
        prev_x , slope = ctx.saved_tensors
        return grad_output *


# Define the Linear Regression Model
class LinearRegressionModel(nn.Module):
    def __init__(self, slope=1):
        super().__init__()
        self.slope = nn.Parameter(torch.ones(1) * slope)

    def forward(self, x):
        # Use the custom LearnedSiLUFunction
        ...

# Initialize the model, loss function, and optimizer
model = LinearRegressionModel()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Training loop
epochs = 1000
for epoch in range(epochs):
    # Forward pass
    predictions = model(X)
    loss = criterion(predictions, y)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Log progress every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")


Epoch [100/1000], Loss: 1.0552
Epoch [200/1000], Loss: 0.8031
Epoch [300/1000], Loss: 0.7150
Epoch [400/1000], Loss: 0.6826
Epoch [500/1000], Loss: 0.6705
Epoch [600/1000], Loss: 0.6659
Epoch [700/1000], Loss: 0.6642
Epoch [800/1000], Loss: 0.6635
Epoch [900/1000], Loss: 0.6632
Epoch [1000/1000], Loss: 0.6632


In [4]:
# Display the learned parameters
[w, b] = model.linear.parameters()
print(f"Learned weight: {w.item():.4f}, Learned bias: {b.item():.4f}")

# Testing on new data
X_test = torch.tensor([[4.0], [7.0]])
with torch.no_grad():
    predictions = model(X_test)
    print(f"Predictions for {X_test.tolist()}: {predictions.tolist()}")

Learned weight: 1.9557, Learned bias: 2.2181
Predictions for [[4.0], [7.0]]: [[11.04088020324707], [16.907970428466797]]
